In [1]:
import os

denoising_weight = [100,150,200]
layers = [3,7]
normalizations = ['clr', 'log_rel_abundance', 'none']
norm_names = {'clr':'clr', 'log_rel_abundance':'log_rel_ab', 'none':'counts'}
emb_style = ['avg-pool', 'cls']
emb_name = {'avg-pool':'pool', 'cls':'cls'}
downsample_ratios = [(0.01,0.7),(0.2,0.8)]
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/configs/pretrain/denoising_contrastive/'
outdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/pretrain/denoising_contrastive/'
os.makedirs(topdir, exist_ok=True)
list_file_name = os.path.join(topdir, 'config_list.txt')
with open(list_file_name, 'w') as list_file:
  for n in normalizations:
    for l in layers:
      for d in denoising_weight:
            for e in emb_style:
              for dr in downsample_ratios:
                basename = f'{norm_names[n]}_{l}layers_wd_{d}_{int(dr[0]*100)}_{int(dr[1]*100)}_{emb_name[e]}'
                text = f'''
                paths:
                  output_dir: "{outdir}/{basename}"
                  model_config_path: "{outdir}/{basename}/model_config.json"
                  ann_table_path: "/project/aip-rahulgk/gutmodel/hmc_final_fixed/pretrain.h5ad"
                wandb:
                  enabled: true
                  entity: "haoze-deng-university-of-toronto"
                  project: "microbiome-foundation"
                  run_name: denoising_contrastive_{basename}
                  run_notes: "Pretraining denoising + contrastive"

                training:
                  seed: 423985693
                  init_lr: 1e-4
                  batch_size: 64
                  max_epochs: 200 # for testing
                  cosine_warmup_ratio_or_step: 0.1
                  log_interval: 10
                  patience: 20
                  grad_accumulation_steps: 1
                  enable_fp16: false
                  # masking_prob: 0.70
                  notes: "Testing contrastive_denoising"
                  tasks: ["denoising", "contrastive"] # contrastive, denoising, denoising_from_token, denoising_dm, denoising_lnm
                  contrastive_temperature: 0.2
                  w_denoising: {d}

                validation:
                  batch_size: 128
                  eval_interval_epochs: 1

                data:
                  norm_strategy: "{n}" # choices: clr, none, rel_abundance, log_rel_abundance, log_counts, binning
                  split_key: "study_id" # we probably want to split train/val by entire studies, so it resembles close to real-world/test scenario
                  val_size: 0.1
                  use_batch_labels: false
                  max_seq_len: 200
                  num_workers: 4
                  downsample_distribution: "multinomial"
                  downsample_ratio_range: [{dr[0]}, {dr[1]}]
                  

                model:
                  
                  tasks:
                    do_mvc: false
                    do_taxa_decoder: false
                    do_contrastive: true

                  # === FLEXIBLE ARCHITECTURE SECTION ===
                  # Any key added here is automatically passed to the model config dict
                  params:
                    d_model: 128
                    d_proj: 128
                    seq_len: 200
                    nhead: 8
                    d_hid: 512
                    nlayers: {l} #3 7
                    dropout: 0.1
                    abundance_emb_style: "continuous" # choices: scaling, continuous, category
                    sample_emb_style: "{e}" # "cls" "avg-pool"
                    # freeze_vocab: false
                    # freeze_value_encoder: false
                    # do_attn_mask: false
                    # Example: If you change architecture, just add new keys here
                    # activation: "gelu" 
                    # use_flash_attn: true
                    use_gnn: false
                    model_distribution: "dm"


                debug:
                  # nrows: null
                  start_over: false
                '''
                fname = f'{basename}.yaml'
                fpath = os.path.join(topdir, fname)
                with open(fpath, 'w') as f:
                  f.write(text)
                list_file.write(fpath + '\n')
      # print(80*"#")
      # print(text)
      

In [10]:
!pwd

/project/6101781/dpellow/gut_microbiome_GPT/notebooks
